# 17 — Climate Forecasting

**Context Demystifies Forecasting**

This notebook moves from toy systems to a climate-style forecasting workflow.

Goal:

> Do latent-state ideas remain useful on realistic environmental time series?

The notebook uses a synthetic climate dataset by default so it runs anywhere. A later revision can swap in the Jena Climate dataset.


## Climate variables

We simulate:

- temperature
- pressure
- humidity

and compare:

1. baseline forecast
2. output-constrained forecast
3. physical/residual decomposition


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

rng = np.random.default_rng(260616076)


In [ ]:
days = np.arange(0, 1460)

temperature = (
    15
    + 10*np.sin(2*np.pi*days/365)
    + 0.8*rng.normal(size=len(days))
)

pressure = (
    1013
    + 8*np.sin(2*np.pi*days/180 + 0.3)
    + 1.5*rng.normal(size=len(days))
)

humidity = (
    60
    - 0.6*(temperature-15)
    + 4*rng.normal(size=len(days))
)

df = pd.DataFrame({
    "day": days,
    "temperature": temperature,
    "pressure": pressure,
    "humidity": humidity,
})

df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(df["day"], df["temperature"])
ax.set_title("Synthetic climate temperature")
ax.set_xlabel("day")
ax.set_ylabel("temperature")
fig.tight_layout()
fig.savefig(FIGURES / "17_temperature_series.png", dpi=160)
plt.show()


## Train / test split

In [ ]:
split = 1100

train = df.iloc[:split]
test = df.iloc[split:]

x_train = train["day"].to_numpy()
x_test = test["day"].to_numpy()

y_train = train["temperature"].to_numpy()
y_test = test["temperature"].to_numpy()


## Baseline forecast

Simple linear extrapolation from recent observations.


In [ ]:
recent = y_train[-30:]
slope = np.polyfit(np.arange(len(recent)), recent, 1)[0]

baseline = recent[-1] + slope * np.arange(1, len(y_test)+1)


## Output-constrained forecast

Forecast first, then clip to plausible temperature range.


In [ ]:
lower = np.quantile(y_train, 0.02)
upper = np.quantile(y_train, 0.98)

output_constrained = np.clip(baseline, lower, upper)


## Physical/residual decomposition

Estimate seasonal structure and separate residual variation.


In [ ]:
X = np.column_stack([
    np.ones_like(x_train),
    np.sin(2*np.pi*x_train/365),
    np.cos(2*np.pi*x_train/365),
])

coef, *_ = np.linalg.lstsq(X, y_train, rcond=None)

X_future = np.column_stack([
    np.ones_like(x_test),
    np.sin(2*np.pi*x_test/365),
    np.cos(2*np.pi*x_test/365),
])

physical_forecast = X_future @ coef


In [ ]:
def rmse(a,b):
    return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))

metrics = pd.DataFrame([
    ["baseline", rmse(y_test, baseline)],
    ["output_constrained", rmse(y_test, output_constrained)],
    ["physical_residual", rmse(y_test, physical_forecast)],
], columns=["method","RMSE"])

metrics


In [ ]:
fig, ax = plt.subplots(figsize=(11,4))

ax.plot(x_test, y_test, label="truth")
ax.plot(x_test, baseline, label="baseline")
ax.plot(x_test, output_constrained, label="output constrained")
ax.plot(x_test, physical_forecast, label="physical/residual")

ax.legend()
ax.set_title("Climate forecasting comparison")

fig.tight_layout()
fig.savefig(FIGURES / "17_climate_forecasting_comparison.png", dpi=160)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))

ax.bar(metrics["method"], metrics["RMSE"])
ax.set_title("Climate forecasting RMSE")

fig.tight_layout()
fig.savefig(FIGURES / "17_climate_rmse.png", dpi=160)
plt.show()


## Interpretation

If seasonal structure explains most variation, a physically meaningful latent state can outperform simple extrapolation.

The next notebooks can replace the synthetic dataset with Jena Climate and examine interpretability and forecast stability on real measurements.
